# Exercise 2. (Modern) Topic Modeling on News
In this exercise, we'll perform topic modeling on *document embeddings* using {cite:t}`kardos_turftopic_2025`'s [Turftopic](https://x-tabdeveloping.github.io/turftopic/) (aka the legend Márton)! 

> Note: Most of this tutorial is based on [Márton´s tutorial](https://x-tabdeveloping.github.io/turftopic/tutorials/arxiv_ml/). Highly recommend reading this after class to get the details :)


## Dataset & RQ
We'll use subset of a BBC news dataset from {cite:t}`li2024latesteval`. 

The news is inherently topical, often reflecting key events in the year. But, when looking back, my memory *can sometimes* exaggerate some stories or forget others entirely. Rather than trusting my memory, let's answer the question computationally:

```{admonition} RESEARCH QUESTION
What kinds of news topics were covered in April 2020 versus April 2025?
```

## 2.1 Install Packages
This time, let's install all necessary packages now:

In [ ]:
%pip install datasets plotly sentence_transformers
%pip install "turftopic[umap-learn,datamapplot]" 
# installing turftopic with extra dependencies requires string format



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2.2 Loading the Data
This time we'll use the `datasets` package from company "HuggingFace":

In [3]:
import datasets

:::{admonition} HANDS-ON
:class: red
Go and look at the [BBC dataset](https://huggingface.co/datasets/RealTimeData/bbc_news_alltime) on HuggingFace:

- On the Dataset Viewer, can you figure out how many subsets of the data there is? 
- What kinds of columns does the dataset have?
:::

We can load snippets as such:

In [4]:
ds_apr20 = datasets.load_dataset(
                            "RealTimeData/bbc_news_alltime", 
                            "2020-04" # subset of the data year: 2020, month: 04 (april)
                            )["train"] # there is only train, and we select this

Note: The `Dataset` class differs a little from a `pandas.DataFrame`, but works quite well with many newer workflows (for working wth `Transformers` which you'll learn about later!)

In [5]:
# print the ds 
print(ds_apr20)

Dataset({
    features: ['title', 'published_date', 'authors', 'description', 'section', 'content', 'link', 'top_image'],
    num_rows: 1152
})


Let's save all text to a `texts_apr20` variable:

In [6]:
texts_apr20 = ds_apr20["content"]
print(texts_apr20[1][:600])  # print first 600 characters of the 1st news article!

An "enormous strain" has been put on the system for obtaining protective kit for NHS staff and care workers, the education secretary has said.

Some 400,000 gowns had been expected to arrive from Turkey on Sunday, but the government said it had been delayed.

Gavin Williamson was asked by the BBC why British suppliers offering to make protective kit had not been contacted.

He responded that government hoped to speak to them within the next 24 hours, and the gowns should arrive on Monday.

"I think we all recognise the enormous strain that has been placed on the whole system and we also recogn


## 2.3 Embeddings with SentenceTransformers

We'll use a `Transformer` model, specifically trained for `embeddings` using the [SentenceTransformers](https://sbert.net/) package:

In [7]:
from sentence_transformers import SentenceTransformer

:::{admonition} LLM FRAMING: TRANSFORMERS? 
:class: fuchsia, dropdown
`Transformers` are the main architecture that almost all modern LLMs are built on. You'll learn more about what they are and how they work in a few weeks!

If you are impatient (:DD), you can read about them in [Jurafsky & Martin (2025), chapter 8](https://web.stanford.edu/~jurafsky/slp3/8.pdf).
:::

Let's initialize a model:

In [8]:
model_name = "all-MiniLM-L6-v2"  # a small, efficient model
encoder = SentenceTransformer(model_name)

We can now extract embeddings:

In [9]:
embeddings_apr20 = encoder.encode(texts_apr20, show_progress_bar=True)

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

## 2.4 Extract Topic Data

We're ready to use the `turftopic` package, using a Top2Vec *clustering* topic model 

In [10]:
from turftopic import Top2Vec

/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/serialization.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [11]:
model = Top2Vec(encoder=encoder, random_state=42)
topic_data = model.prepare_topic_data(texts_apr20, embeddings=embeddings_apr20)

Output()

[17:48:36] Term extraction done.                                                                     ]8;id=49693;file:///Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/models/cluster.py\cluster.py]8;;\:]8;id=254477;file:///Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/models/cluster.py#481\481]8;;\

/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: 
UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(

[17:48:40] Dimensionality reduction done.                                                            ]8;id=818017;file:///Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/models/cluster.py\cluster.py]8;;\:]8;id=808577;file:///Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/models/cluster.py#490\490]8;;\

           Clustering done.                                                                          ]8;id=946283;file:///Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/models/cluster.py\cluster.py]8;;\:]8;id=676804;file:///Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/models/cluster.py#493\493]8;;\

[17:48:41] Parameter estimation done.                                                                ]8;id=974340;file:///Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/models/cluster.py\cluster.py]8;;\:]8;id=856205;file:///Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/models/cluster.py#512\512]8;;\

           Model fitting done.                                                                       ]8;id=450540;file:///Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/models/cluster.py\cluster.py]8;;\:]8;id=402962;file:///Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/turftopic/models/cluster.py#526\526]8;;\

### 2.5 Looking at the topics!
Let's print all topics:

In [12]:
model.print_topics()

┏━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Topic ID ┃ Highest Ranking                                                                                      ┃
┡━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       -1 │ pandemics, nhs, pandemic, coronavirus, covid, illnesses, epidemic, emergencies, outbreak, illness    │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        0 │ video, unable, videos, play, impossible, youtube, couldn, plays, played, permitted                   │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │ ships, ship, crew, crews, boat, navy, vessel, stranded, boats, passengers                            │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        2 │ supermarkets, retailer, supermarket, retailers, customers, retail, shoppers, shop, shopping, shops   │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        3 │ banking, banks, nhs, lenders, bank, recession, financial, insurance, loan, airlines                  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        4 │ footballers, sporting, liverpool, tottenham, itv, gareth, players, injured, sports, nhs              │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        5 │ recession, gdp, economists, economy, economist, pandemic, economic, forecasts, crisis, pandemics     │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        6 │ nhs, students, education, schools, educational, teachers, children, programmes, compulsory,          │
│          │ assessments                                                                                          │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        7 │ charities, charity, fundraising, nhs, funding, donations, charitable, donating, aid, taxpayer        │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        8 │ smartphone, covid, coronavirus, quarantined, twitter, mobile, pandemics, virus, quarantine, privacy  │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        9 │ corbyn, candidate, chancellor, secretary, minister, presenter, politicians, mps, election, brexit    │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       10 │ nhs, precautions, masks, clinicians, pandemics, patients, hospitals, policies, emergencies, pandemic │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       11 │ tom, moore, nhs, gareth, clarke, charity, kent, bbcnewsents, ward, cheshire                          │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       12 │ bbcnewsents, celebrating, nhs, queen, briefing, presenter, honour, emergencies, briefings, charity   │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       13 │ comedian, singer, concert, presenter, songs, famous, richard, itv, michael, mike                     │
├──────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────┤
│       14 │ coronavirus, nhs, pandemics, clinicians, co

:::{admonition} QUESTION
:class: red
What topics seems to be predominant? Can you give "titles" to the 5 highest ranked topics?
:::

:::{admonition} Dealing with similar topics?
:class: tip, dropdown
Some article groups share very similar keywords (e.g., topics 20 & 21). 

There are ways to merge topics which we won't cover today. See ["Building a Topic Hierarchy"](https://x-tabdeveloping.github.io/turftopic/tutorials/arxiv_ml/#interpreting-results) if interested.
:::


### Your Turn: Run the workflow on 2025!
:::{admonition} HANDS-ON
:class: red
To practice the workflow, run it in your own notebook using the April 2025 subset:
- Load the dataset `ds_apr25`  
- Extract embeddings from `texts_apr25`  
- Run the topic model  
- Consider how the topics differ from 2017!
:::

## 2.6 Explore TurfTopic's other tutorials
There are loads of diff. ways to approach topic modeling! 

We have just done the `Cluster Analysis` tutorial from TurfTopic, but there are three others on TurfTopic's [Tutorial Overview](https://x-tabdeveloping.github.io/turftopic/tutorials/overview/):  

:::{admonition} HANDS-ON
:class: red
1. Skim through the other 3 tutorials: [Discourse Analysis](https://x-tabdeveloping.github.io/turftopic/tutorials/religious/), [Dimensional Analysis](https://x-tabdeveloping.github.io/turftopic/tutorials/ideologies/), and [Dissatisfaction Analysis](https://x-tabdeveloping.github.io/turftopic/tutorials/reviews/).  
2. Consider whether there is an RQ that can be answered with methods from the tutorials above using our news data.  
3. Implement parts of one of the tutorials to address this RQ (or just play around with one of the tutorials!)
:::